# 🔩 Fastener Company Finder — Google Colab

Tool tổng hợp **website + email** các công ty **sản xuất / nhập khẩu / phân phối** fasteners, screws, threaded rod, studs, washers... ở **Mỹ và Châu Âu**.

Notebook này chỉ là **adapter mỏng**: cell đầu tự tải bản core mới nhất (`fastener_finder.py`) từ repo GitHub [nghiant96/fastener-contact-tool](https://github.com/nghiant96/fastener-contact-tool) — sửa lỗi/nâng cấp ở core là notebook tự có, không cần đổi link.

**Cách dùng:** chạy lần lượt các cell từ trên xuống. Mỗi công ty được chấm điểm `qualification_status` (qualified / review / rejected) + `confidence_score` để bạn biết dòng nào tin được.

In [ ]:
#@title 1️⃣ Cài thư viện + tải core mới nhất từ GitHub
%pip -q install ddgs pandas openpyxl requests

import importlib, urllib.request
CORE_URL = ("https://raw.githubusercontent.com/nghiant96/"
            "fastener-contact-tool/main/fastener_finder.py")
urllib.request.urlretrieve(CORE_URL, "fastener_finder.py")
import fastener_finder as ff
importlib.reload(ff)
print("✅ Đã nạp core:", ff.__doc__.strip().splitlines()[0])

In [ ]:
#@title 2️⃣ (Tuỳ chọn) Chỉnh cấu hình — bỏ qua nếu dùng mặc định
# Ví dụ: thêm sản phẩm / quốc gia
# ff.PRODUCTS.append("anchor bolts")
# ff.REGIONS["Czech"] = (["Czech Republic"], "cz-cs")
# ff.RESULTS_PER_QUERY = 20
print(f"Sản phẩm: {ff.PRODUCTS}\nVai trò: {ff.ROLES}\nQuốc gia: {list(ff.REGIONS)}")

In [ ]:
#@title 3️⃣ Quét 1 lần (~870 truy vấn / 29 nước, 45–75 phút) rồi tải file về
# Mẹo: muốn nhanh, thu hẹp ff.REGIONS / ff.PRODUCTS ở cell 2,
# hoặc dùng chế độ chạy liên tục bên dưới (mỗi vòng chỉ 60 truy vấn).
df = ff.run(out_csv="fastener_companies.csv")

from google.colab import files
files.download("fastener_companies.csv")
files.download("fastener_companies.xlsx")
df.head(20)

## 📧 Tìm EMAIL trên các website đã thu thập

Vào từng website, đọc trang chủ + tối đa 3 trang liên hệ (contact / kontakt / impressum...), trích tối đa 5 email/công ty — bắt được cả email bị Cloudflare che và dạng `name (at) domain (dot) com`.

Kết quả tách rõ: `emails` (cùng domain / liên quan — đáng tin), `emails_external` (email lạ), `email_status` (found / not_found / timeout / blocked / error) và `email_found_on` (các trang tìm thấy). **Chạy lại cell là tự resume** — chỉ quét website chưa xong.

In [ ]:
#@title 📧 Quét email rồi tải file về
df2 = ff.add_emails_to_csv("fastener_companies.csv")

from google.colab import files
files.download("fastener_companies_with_emails.csv")
files.download("fastener_companies_with_emails.xlsx")

## 🔄 (Tuỳ chọn) Chế độ CHẠY LIÊN TỤC

Quét lặp vô hạn: mỗi vòng ~60 truy vấn ngẫu nhiên (từ khoá xoay vòng), **chỉ thêm công ty MỚI** vào file tổng trên **Google Drive** — Colab ngắt kết nối cũng không mất dữ liệu, mở lại chạy tiếp là tích luỹ tiếp.

⚠️ Colab miễn phí tự ngắt sau ~90 phút không tương tác (tối đa 12h). Chạy 24/7 thật sự: dùng bản standalone trên máy — `python fastener_finder.py --loop 30`.

In [ ]:
#@title 🔄 Chạy liên tục (dừng bằng nút ⏹ Stop — dữ liệu lưu Drive sau mỗi vòng)
from google.colab import drive
drive.mount("/content/drive")

ff.run_forever(
    interval_minutes=15,   # nghỉ giữa các vòng
    queries_per_cycle=60,  # số truy vấn mỗi vòng
    master_csv="/content/drive/MyDrive/fastener_companies_master.csv",
)

## 💡 Mẹo
- **Cột chất lượng**: ưu tiên dòng `qualification_status = qualified`, sau đó duyệt tay nhóm `review`; `confidence_score` càng cao càng tin. `verified_country` là nước suy từ đuôi tên miền (.de, .fr...).
- **Chấm điểm lại file cũ**: `ff.qualify_dataframe(df)` hoặc CLI `python fastener_finder.py --requalify file.csv`.
- **Thêm quốc gia**: `ff.REGIONS["Czech"] = (["Czech Republic"], "cz-cs")` ở cell 2.
- **Bị chặn / ratelimit**: `ff.SLEEP_RANGE = (3, 6)` rồi chạy lại.
- Kết quả từ search engine vẫn nên kiểm tra tay trước khi gửi email hàng loạt.